In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

gapgpt_base_url = os.getenv("GAPGPT_BASE_URL")
gapgpt_api_key = os.getenv("GAPGPT_API_KEY")


llm = ChatOpenAI(
    model = "gpt-5.6-luna",
    base_url= gapgpt_base_url,
    api_key = gapgpt_api_key
)


respond = llm.invoke("salam, halet chetore?")
print(respond.content)

سلام! ممنون، خوبم 😊 تو چطوری؟


In [ ]:
from state import RAGState
from langchain_core.messages import AIMessage, ToolMessage
from output_schemas import _GroundednessVerdict
from models import lite_model
from prompts import _GROUNDEDNESS_PROMPT
from langchain.agents.middleware import after_model
from typing import cast

In [ ]:
@after_model
def response_groumdedness(state, runtime):
    
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage) or last_message.tool_calls:
        return None
    answer_text = str(last_message.content)
    
    tool_context = "\n\n".join(
        str(m.content)
        for m in state["messages"]
        if isinstance(m, ToolMessage)
    )
    if not tool_context:
        return None
    
    grader = lite_model.with_structured_output(_GroundednessVerdict)
    prompt = _GROUNDEDNESS_PROMPT
    chain = prompt|grader
    
    verdict = cast(_GroundednessVerdict, chain.invoke({"context": tool_context, "answer": answer_text}))
    if verdict.binary_score == "yes":
        return None
    
    disclaimer = AIMessage(content="I found a possible answer, but couldn't verify it's fully supported by the retrieved information, so I'm not confident enough to state it.", id=last_message.id)
    return {"messages": [disclaimer]}